/workspace/KG-known0/260410_전체사이클_빈도상위10고정_gpt4_38개rel_IDPPA변형_entity

In [1]:
import pandas as pd
multi_predicate_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_full_binary_question_result.csv')
multi_predicate_df.head(2)

,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,...,associated_with_region,artist_origin_city,reorganized_after_loss_of_papal_states,held_diplomatic_position_in_location,partnership_location,diplomatic_location_request,affiliated_with_location,participant_at_event_location,artist_origin_location,inventor_location_use
0,train_P1877_557,after a work by,P1877,"After reading Alan Glynn 's "" The Dark Fields ...",limitless,alan glynn,14,14,2,3,...,0,0,0,0,0,0,0,0,0,0
1,train_P1877_158,after a work by,P1877,"It was a sequel to the 1952 film "" Heidi "" , w...",heidi,johanna spyri,9,9,25,26,...,0,0,0,0,0,0,0,0,0,0


In [2]:
import spacy
import pandas as pd


nlp = spacy.load("en_core_web_sm")

multi_df = multi_predicate_df.copy()


multi_df["predicate"] = [[] for _ in range(len(multi_df))]


def find_entity_span(doc, entity_text):
    """
    doc 안에서 entity_text와 정확히 일치하는 연속 span을 찾아 반환
    없으면 None 반환
    """
    if not isinstance(entity_text, str) or not entity_text.strip():
        return None

    entity_text = entity_text.strip().lower()
    entity_words = entity_text.split()
    doc_words = [token.text.lower() for token in doc]

    n = len(entity_words)
    for i in range(len(doc_words) - n + 1):
        if doc_words[i:i+n] == entity_words:
            return doc[i:i+n]

    return None


def get_connected_verbs_from_span(span):
    """
    엔티티 span의 각 토큰으로부터 head 방향으로 올라가며
    처음 만나는 VERB/AUX를 predicate 후보로 수집
    """
    predicates = set()

    if span is None:
        return []

    for token in span:
        current = token
        visited = set()

        while current is not None and current.i not in visited:
            visited.add(current.i)

            if current.pos_ in {"VERB", "AUX"}:
                predicates.add(current.lemma_.lower())
                break

            if current.head == current:
                break

            current = current.head

    return list(predicates)


def extract_predicates_for_entities(text, subject, object_):
    """
    subject, object 각각과 관련된 술어를 추출
    결과: list[str]
    """
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)

    subj_span = find_entity_span(doc, subject)
    obj_span = find_entity_span(doc, object_)

    predicates = set()

    for p in get_connected_verbs_from_span(subj_span):
        predicates.add(p)

    for p in get_connected_verbs_from_span(obj_span):
        predicates.add(p)

    return list(predicates)



multi_df["predicate"] = multi_df.apply(
    lambda row: extract_predicates_for_entities(
        row["text"],
        row["subject"],
        row["object"]
    ),
    axis=1
)

In [3]:
multi_df.head()

,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,...,artist_origin_city,reorganized_after_loss_of_papal_states,held_diplomatic_position_in_location,partnership_location,diplomatic_location_request,affiliated_with_location,participant_at_event_location,artist_origin_location,inventor_location_use,predicate
0,train_P1877_557,after a work by,P1877,"After reading Alan Glynn 's "" The Dark Fields ...",limitless,alan glynn,14,14,2,3,...,0,0,0,0,0,0,0,0,0,"[write, read]"
1,train_P1877_158,after a work by,P1877,"It was a sequel to the 1952 film "" Heidi "" , w...",heidi,johanna spyri,9,9,25,26,...,0,0,0,0,0,0,0,0,0,[be]
2,train_P1877_148,after a work by,P1877,It is a loose adaptation of the eponymous Jame...,film,ian fleming,10,10,24,25,...,0,0,0,0,0,0,0,0,0,"[be, inspire]"
3,train_P1877_321,after a work by,P1877,The film is a remake of the 1975 Disney live -...,escape to witch mountain,alexander key,14,17,32,33,...,0,0,0,0,0,0,0,0,0,"[be, base]"
4,train_P1877_63,after a work by,P1877,Balu Mahendra had read the Tamil translation o...,misery,stephen king,14,14,8,9,...,0,0,0,0,0,0,0,0,0,[read]


In [4]:
multi_df.to_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_full_binary_question_result_predicate.csv', index = False)

# predicate 통계

In [5]:
import ast

def count_predicate_items(x):
    
    if isinstance(x, list):
        return len(x)
    
    
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed)
        except Exception:
            pass
    
    
    return 0


multi_df["predicate_count"] = multi_df["predicate"].apply(count_predicate_items)


print(multi_df["predicate_count"].describe())


print(multi_df["predicate_count"].value_counts().sort_index())

count    4000.000000
mean        1.244500
std         0.490693
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
Name: predicate_count, dtype: float64
predicate_count
0      95
1    2848
2    1042
3      14
4       1
Name: count, dtype: int64
